# CSBrain Embedding Visualization -- BCIC-IV-2a

This notebook downloads the BCIC-IV-2a dataset, preprocesses it into LMDB format,
loads the fine-tuned CSBrain model, extracts backbone embeddings, reduces them to 3D
using t-SNE, and visualizes clusters by class label using Plotly.

In [1]:
import sys
import os
import ssl
import argparse
import urllib.request
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.manifold import TSNE
import plotly.express as px
import pandas as pd
import scipy.io
import lmdb
import pickle
from scipy.signal import butter, lfilter, resample

# Disable SSL verification for downloading dataset
ssl._create_default_https_context = ssl._create_unverified_context

KeyboardInterrupt: 

## Configuration

In [ ]:
WEIGHTS_PATH = "model_weights/epoch5_acc_0.57726_kappa_0.43634_f1_0.56665.pth"
RAW_DATA_DIR = "data/BCICIV2a/raw"
LMDB_DIR = "data/BCICIV2a/processed_lmdb"

NUM_CLASSES = 4
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64

# BCIC-IV-2a class labels (4-class motor imagery)
CLASS_NAMES = {0: "Left Hand", 1: "Right Hand", 2: "Both Feet", 3: "Tongue"}

print(f"Using device: {DEVICE}")

Using device: cpu


## Download BCIC-IV-2a Raw Data

Downloads the 9 subjects `.mat` files (evaluation + training) from the BNCI Horizon 2020 server.

In [ ]:
os.makedirs(RAW_DATA_DIR, exist_ok=True)

BASE_URL = "http://bnci-horizon-2020.eu/database/data-sets/001-2014"
SUBJECTS = [f"A{i:02d}" for i in range(1, 10)]  # A01 to A09
SUFFIXES = ["T", "E"]  # Training and Evaluation

for subj in SUBJECTS:
    for suffix in SUFFIXES:
        fname = f"{subj}{suffix}.mat"
        fpath = os.path.join(RAW_DATA_DIR, fname)
        if os.path.exists(fpath):
            print(f"Already exists: {fname}")
            continue
        url = f"{BASE_URL}/{fname}"
        print(f"Downloading {fname}...")
        try:
            urllib.request.urlretrieve(url, fpath)
            print(f"  Saved to {fpath}")
        except Exception as e:
            print(f"  Failed: {e}")

print(f"\nFiles in {RAW_DATA_DIR}:")
print(sorted(os.listdir(RAW_DATA_DIR)))

Already exists: A01T.mat
Already exists: A01E.mat
Already exists: A02T.mat
Already exists: A02E.mat
Already exists: A03T.mat
Already exists: A03E.mat
Already exists: A04T.mat
Already exists: A04E.mat
Already exists: A05T.mat
Already exists: A05E.mat
Already exists: A06T.mat
Already exists: A06E.mat
Already exists: A07T.mat
Already exists: A07E.mat
Already exists: A08T.mat
Already exists: A08E.mat
Already exists: A09T.mat
Already exists: A09E.mat

Files in data/BCICIV2a/raw:
['A01E.mat', 'A01T.mat', 'A02E.mat', 'A02T.mat', 'A03E.mat', 'A03T.mat', 'A04E.mat', 'A04T.mat', 'A05E.mat', 'A05T.mat', 'A06E.mat', 'A06T.mat', 'A07E.mat', 'A07T.mat', 'A08E.mat', 'A08T.mat', 'A09E.mat', 'A09T.mat']


## Preprocess into LMDB

Applies bandpass filter (0.3-50 Hz), crops the motor imagery window (2-6s), resamples to 800 points,
and reshapes to `(22, 4, 200)`. Follows the same preprocessing as `CBraMod/preprocessing_bciciv2a.py`.

In [ ]:
def butter_bandpass(low_cut, high_cut, fs, order=5):
    nyq = 0.5 * fs
    low = low_cut / nyq
    high = high_cut / nyq
    b, a = butter(order, [low, high], btype="band")
    return b, a

if os.path.exists(os.path.join(LMDB_DIR, "data.mdb")):
    print(f"LMDB already exists at {LMDB_DIR}, skipping preprocessing.")
else:
    os.makedirs(LMDB_DIR, exist_ok=True)

    files_dict = {
        "train": ["A01E.mat", "A01T.mat", "A02E.mat", "A02T.mat", "A03E.mat", "A03T.mat",
                  "A04E.mat", "A04T.mat", "A05E.mat", "A05T.mat"],
        "val": ["A06E.mat", "A06T.mat", "A07E.mat", "A07T.mat"],
        "test": ["A08E.mat", "A08T.mat", "A09E.mat", "A09T.mat"],
    }

    dataset_keys = {"train": [], "val": [], "test": []}
    db = lmdb.open(LMDB_DIR, map_size=1610612736)

    for split in files_dict:
        for file in files_dict[split]:
            fpath = os.path.join(RAW_DATA_DIR, file)
            if not os.path.exists(fpath):
                print(f"WARNING: {fpath} not found, skipping.")
                continue
            print(f"Processing {file} ({split})...")
            data = scipy.io.loadmat(fpath)
            num = len(data["data"][0])

            for j in range(3, num):
                raw_data = data["data"][0, j][0, 0][0][:, :22]
                events = data["data"][0, j][0, 0][1][:, 0]
                labels = data["data"][0, j][0, 0][2][:, 0]
                length = raw_data.shape[0]
                events = events.tolist()
                events.append(length)

                annos = []
                for i in range(len(events) - 1):
                    annos.append((events[i], events[i + 1]))

                for i, (anno, label) in enumerate(zip(annos, labels)):
                    sample = raw_data[anno[0]:anno[1]].transpose(1, 0)
                    sample = sample - np.mean(sample, axis=0, keepdims=True)
                    b, a = butter_bandpass(0.3, 50, 250)
                    sample = lfilter(b, a, sample, -1)
                    sample = sample[:, 2 * 250:6 * 250]
                    sample = resample(sample, 800, axis=-1)
                    sample = sample.reshape(22, 4, 200)

                    sample_key = f"{file[:-4]}-{j}-{i}"
                    data_dict = {"sample": sample, "label": label - 1}
                    txn = db.begin(write=True)
                    txn.put(key=sample_key.encode(), value=pickle.dumps(data_dict))
                    txn.commit()
                    dataset_keys[split].append(sample_key)

    txn = db.begin(write=True)
    txn.put(key="__keys__".encode(), value=pickle.dumps(dataset_keys))
    txn.commit()
    db.close()

    for split in dataset_keys:
        print(f"  {split}: {len(dataset_keys[split])} samples")
    print("Preprocessing complete!")

LMDB already exists at data/BCICIV2a/processed_lmdb, skipping preprocessing.


## Load Dataset

In [ ]:
from datasets.bciciv2a_dataset import CustomDataset

params = argparse.Namespace(
    datasets_dir=LMDB_DIR,
    use_SmallerToken=False,
    batch_size=BATCH_SIZE,
    num_of_classes=NUM_CLASSES,
    model="CSBrain",
    use_pretrained_weights=False,
    foundation_dir="",
    cuda=0,
    dropout=0.3,
    n_layer=12,
)

test_set = CustomDataset(LMDB_DIR, params, mode="test")
test_loader = DataLoader(
    test_set,
    batch_size=BATCH_SIZE,
    collate_fn=test_set.collate,
    shuffle=False,
)
print(f"Test set size: {len(test_set)}")

Test set size: 1152


## Load Fine-tuned Model

In [ ]:
from models.model_for_bciciv2a import Model

model = Model(params)

# Load fine-tuned weights
state_dict = torch.load(WEIGHTS_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)
model = model.to(DEVICE)
model.eval()
print("Model loaded successfully.")

Sorted Indices: [0, 18, 19, 20, 21, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
Model loaded successfully.


## Extract Backbone Embeddings

We extract embeddings from the backbone output (before the classification head).
The backbone outputs shape `(batch, 22, 4, 200)` which gets flattened to `(batch, 17600)`.

In [ ]:
all_embeddings = []
all_labels = []

with torch.no_grad():
    for data, labels in test_loader:
        data = data.to(DEVICE)
        bz, ch_num, seq_len, patch_size = data.shape

        # Get backbone embeddings (before classification head)
        feats = model.backbone(data)
        feats = feats.contiguous().view(bz, -1)  # (batch, 22*4*200)

        all_embeddings.append(feats.cpu().numpy())
        all_labels.append(labels.numpy())

all_embeddings = np.concatenate(all_embeddings, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

print(f"Embeddings shape: {all_embeddings.shape}")
print(f"Labels shape: {all_labels.shape}")
print(f"Label distribution: {dict(zip(*np.unique(all_labels, return_counts=True)))}")

Embeddings shape: (1152, 17600)
Labels shape: (1152,)
Label distribution: {np.int64(0): np.int64(288), np.int64(1): np.int64(288), np.int64(2): np.int64(288), np.int64(3): np.int64(288)}


## Reduce to 3D with t-SNE

In [ ]:
tsne = TSNE(n_components=3, random_state=42, perplexity=30) # n_iter=1000)
embeddings_3d = tsne.fit_transform(all_embeddings)
print(f"3D embeddings shape: {embeddings_3d.shape}")

3D embeddings shape: (1152, 3)


## Interactive 3D Scatter Plot with Plotly

In [ ]:
df = pd.DataFrame({
    "t-SNE 1": embeddings_3d[:, 0],
    "t-SNE 2": embeddings_3d[:, 1],
    "t-SNE 3": embeddings_3d[:, 2],
    "Label": [CLASS_NAMES[int(l)] for l in all_labels],
})

fig = px.scatter_3d(
    df,
    x="t-SNE 1",
    y="t-SNE 2",
    z="t-SNE 3",
    color="Label",
    title="CSBrain Embeddings - BCIC-IV-2a (3D t-SNE)",
    opacity=0.7,
    width=900,
    height=700,
)
fig.update_traces(marker=dict(size=4))
fig.show()